# Klike — Recommendations Engine interativo

**Parte 3 — Demonstração funcional da Millai / Klike**

Informe uma campanha nova sem editar o código e clique em **Gerar recomendações**. O motor utiliza o Ridge reduzido e associações observacionais calculadas apenas no treino.

**Instruções:** selecione seu ambiente Python, execute as três células de código abaixo em ordem e utilize o formulário. O GitHub mostra a fonte, mas o formulário requer JupyterLab ou VS Code.

> Diferenças entre previsões e entre grupos observacionais não demonstram causalidade. Nem todas as campanhas terão três alterações elegíveis.


In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data" / "klike_challenge_dataset.csv").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "data" / "klike_challenge_dataset.csv").is_file():
    raise FileNotFoundError("Execute a partir da raiz do projeto ou de notebooks/.")
sys.path.insert(0, str(PROJECT_ROOT))
from src.recommendations_engine import RecommendationsEngine
engine = RecommendationsEngine(PROJECT_ROOT)
print(f"Ridge carregado. Referência contextual: {len(engine.df_train)} campanhas de treinamento.")


## Configure a campanha

As dez entradas abaixo correspondem às features do Ridge reduzido. As opções categóricas vêm do conjunto de treinamento. A análise pode levar alguns segundos devido às reamostragens bootstrap.


In [ ]:
import ipywidgets as widgets
from html import escape
from IPython.display import display, HTML, clear_output

style = {"description_width": "155px"}
wide = widgets.Layout(width="390px")
campaign_id = widgets.Text(value="CAMPANHA-NOVA", description="Identificador", style=style, layout=wide)
platform = widgets.Dropdown(options=engine.categorias_validas["platform"], value="Meta", description="Plataforma", style=style, layout=wide)
category = widgets.Dropdown(options=engine.categorias_validas["category"], value="E-commerce", description="Categoria", style=style, layout=wide)
video_duration_s = widgets.BoundedIntText(value=16, min=1, max=10000, description="Duração (s)", style=style, layout=wide)
video_format = widgets.Dropdown(options=engine.categorias_validas["format"], value="vertical", description="Formato", style=style, layout=wide)
text_density = widgets.Dropdown(options=engine.categorias_validas["text_density"], value="low", description="Densidade de texto", style=style, layout=wide)
is_retargeting = widgets.Checkbox(value=False, description="Retargeting", indent=False)
has_subtitle = widgets.Checkbox(value=False, description="Legendas", indent=False)
has_cta = widgets.Checkbox(value=False, description="CTA", indent=False)
has_hook = widgets.Checkbox(value=False, description="Hook", indent=False)
has_face = widgets.Checkbox(value=False, description="Rosto humano", indent=False)
button = widgets.Button(description="Gerar recomendações", button_style="success", icon="check")
output = widgets.Output(layout=widgets.Layout(border="1px solid #cccccc", padding="12px"))
form = widgets.VBox([
    widgets.HTML("<h3>Configurações</h3>"),
    campaign_id,platform,category,video_duration_s,video_format,text_density,
    widgets.HTML("<h3>Características do criativo</h3>"),
    widgets.HBox([has_hook,has_face]),widgets.HBox([has_subtitle,has_cta]),
    is_retargeting,button,output
])
display(form)


In [ ]:
def on_recommend_click(_):
    button.disabled = True
    try:
        with output:
            clear_output(wait=True)
            print("Calculando previsões e associações...")
        campaign = {
            "campaign_id": campaign_id.value.strip() or "CAMPANHA-NOVA",
            "platform":platform.value, "category":category.value,
            "is_retargeting":is_retargeting.value,
            "video_duration_s":video_duration_s.value,
            "format":video_format.value,
            "has_subtitle":has_subtitle.value, "has_cta":has_cta.value,
            "has_hook":has_hook.value, "has_face":has_face.value,
            "text_density":text_density.value
        }
        result = engine.recomendar_campanha(campaign, max_recomendacoes=3, delta_minimo=1.0, n_bootstrap=2000)
        with output:
            clear_output(wait=True)
            display(HTML(f"<h3>Campanha {escape(result['campaign_id'])}</h3>"))
            print(f"Klike Score previsto: {result['score_original']:.2f} pontos")
            print(f"Cenários avaliados: {result['total_cenarios']} | Recomendações elegíveis: {result['total_recomendacoes']}")
            for r in result["recomendacoes"]:
                display(HTML(f"<h4>{r['prioridade']}ª recomendação</h4>"))
                print(r["texto"])
                print(f"Grupos observados: {r['n_original']} vs. {r['n_proposto']} campanhas")
            if result["total_recomendacoes"] < 3:
                print("Há menos de três alterações elegíveis segundo os critérios atuais.")
            if not result["avaliacao"].empty:
                display(HTML("<h4>Diagnóstico dos cenários</h4>"))
                display(result["avaliacao"][["Feature","Original","Proposto","Delta Ridge","Status decisão"]].round(3))
            display(HTML("<small>Previsões e associações são observacionais; não representam efeitos causais.</small>"))
    except (ValueError,TypeError,FileNotFoundError) as exc:
        with output:
            clear_output(wait=True)
            print(f"Entrada inválida ou falha de análise: {exc}")
    finally:
        button.disabled = False

button.on_click(on_recommend_click)
print("Interface pronta: preencha os campos e clique em Gerar recomendações.")


## Notas de interpretação

A diferença prevista no Klike Score resulta da alteração isolada de uma característica, mantendo as demais fixas no Ridge reduzido. As evidências contextuais usam somente o treino e intervalos bootstrap exploratórios. Os limiares de 10 observações por grupo e delta mínimo de 1 ponto são heurísticos, não garantias de confiabilidade ou efeito causal.

**Metodologia, validações e conclusões:** veja [parte-3.ipynb](parte-3.ipynb).
